# MENTORA — Train M5: CV/Transcript NER Extractor (BERT-base-uncased)

Target: **Entity F1 >= 0.82** on the gold test split.

**Schema note:** Phase 3 only produced `labeled/m5/gold_val_test.jsonl` (20 hand-authored, unambiguous resumes used directly as gold — no separate silver/auto-tagged training set yet, see datasets/SOURCES.md). This notebook splits gold 60/20/20 into train/val/test instead of the doc's silver-train/gold-val-test split. **TODO once real Kaggle resumes are added:** regenerate with a real silver set and switch back to train-on-silver, evaluate-on-gold, per the original Phase 3/4 design — that's the setup that gives a trustworthy F1 that isn't just the auto-tagger agreeing with itself.

## 0. Shared setup — mount Drive, confirm GPU, install deps
Run these three cells first in every notebook (per Phase 4 §0).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
# Expect to see "Tesla T4" — if this errors or shows no GPU, go to
# Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save, then re-run.

In [ ]:
import os
DATA = '/content/drive/MyDrive/mentora_data'
MODELS = '/content/drive/MyDrive/mentora_models'
for m in ['model1_gap_classifier', 'model2_question_generator',
          'model3_trajectory_predictor', 'model4_career_matcher', 'model5_cv_ner']:
    os.makedirs(f'{MODELS}/{m}', exist_ok=True)

# NOTE: upload the repo's datasets/ folder to Google Drive at this path first
# (mentora_data/{raw,processed,labeled}/...) — see datasets/README.md.

In [ ]:
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121  # Colab GPU wheel — do NOT reuse the CPU wheel from the backend's laptop plan
!pip install -q transformers peft accelerate sentence-transformers datasets evaluate seqeval spacy sacrebleu rouge_score

## 1. Load and split the gold set (see schema note above)

In [ ]:
import json, random
from datasets import Dataset

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

gold = load_jsonl(f'{DATA}/labeled/m5/gold_val_test.jsonl')

all_tags = sorted({tag for row in gold for tag in row['ner_tags']})
tag2id = {t: i for i, t in enumerate(all_tags)}
id2tag = {i: t for t, i in tag2id.items()}

random.seed(42)
random.shuffle(gold)
n = len(gold)
train, val, test = gold[:int(n*0.6)], gold[int(n*0.6):int(n*0.8)], gold[int(n*0.8):]
print(f'{len(train)} train / {len(val)} val / {len(test)} test resumes')

train_ds = Dataset.from_list(train)
val_ds = Dataset.from_list(val)
test_ds = Dataset.from_list(test)

## 2. Tokenize with label alignment

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize_and_align(batch):
    tokenized = tokenizer(batch['tokens'], truncation=True, is_split_into_words=True, padding='max_length', max_length=256)
    all_labels = []
    for i, tags in enumerate(batch['ner_tags']):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids, prev_word = [], None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word:
                label_ids.append(tag2id[tags[word_id]])
            else:
                label_ids.append(-100)
            prev_word = word_id
        all_labels.append(label_ids)
    tokenized['labels'] = all_labels
    return tokenized

train_ds = train_ds.map(tokenize_and_align, batched=True)
val_ds = val_ds.map(tokenize_and_align, batched=True)
test_ds = test_ds.map(tokenize_and_align, batched=True)

## 3. Model, metrics (entity F1 via seqeval), training

In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer
import evaluate, numpy as np, glob

model = AutoModelForTokenClassification.from_pretrained(
    'bert-base-uncased', num_labels=len(all_tags), id2label=id2tag, label2id=tag2id
)
seqeval = evaluate.load('seqeval')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=2)
    true_preds, true_labels = [], []
    for pred_row, label_row in zip(preds, labels):
        p_seq, l_seq = [], []
        for p, l in zip(pred_row, label_row):
            if l != -100:
                p_seq.append(id2tag[p]); l_seq.append(id2tag[l])
        true_preds.append(p_seq); true_labels.append(l_seq)
    results = seqeval.compute(predictions=true_preds, references=true_labels)
    return {'entity_f1': results['overall_f1']}

OUT_DIR = f'{MODELS}/model5_cv_ner'
args = TrainingArguments(
    output_dir=OUT_DIR, per_device_train_batch_size=8, num_train_epochs=4,
    save_strategy='epoch', eval_strategy='epoch', load_best_model_at_end=True,
    metric_for_best_model='entity_f1', fp16=True,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics)

existing = sorted(glob.glob(f'{OUT_DIR}/checkpoint-*'))
trainer.train(resume_from_checkpoint=existing[-1] if existing else None)

## 4. Save + evaluate on held-out gold test

In [ ]:
trainer.save_model(f'{OUT_DIR}/final')
tokenizer.save_pretrained(f'{OUT_DIR}/final')
print('Test set (gold, held out):', trainer.evaluate(test_ds))

## Target metric: Entity F1 >= 0.82

If it plateaus lower:
- Check per-entity-type F1 (`seqeval.compute(..., mode='strict')`) — usually one type (often CERT, smallest vocab) drags the average down
- Scale up the resume corpus and, once real Kaggle resumes are added, restore the silver/gold split described in the schema note above